In [1]:
import chromadb

from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI

c:\Users\ADMIN\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# === Configuration ===
EMBEDDING_MODEL = "keepitreal/vietnamese-sbert"
MODEL = "deepseek/deepseek-r1-0528-qwen3-8b"
CHROMA_HOST = "localhost"
CHROMA_PORT = 8000
CHROMA_COLLECTION = "tax_documentation"

In [3]:
# === Load FAISS vectorstore ===
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cuda"}
)

client = chromadb.HttpClient(
    host=CHROMA_HOST,
    port=CHROMA_PORT
)

vectorstore = Chroma(
    client=client,
    collection_name=CHROMA_COLLECTION,
    embedding_function=embeddings
)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17764\3884301752.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
W1126 22:53:41.716000 17764 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_17764\3884301752.py:12: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [4]:
llm = ChatOpenAI(
    model=MODEL,  # example: "llama-3.1-8b-instruct"
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",  # can be anything, LM Studio ignores it
    temperature=0,
    max_tokens=256,
)

In [5]:
def rag_query(question: str, k: int = 5) -> str:
    docs = vectorstore.similarity_search(question, k=k)
    context_texts = []
    for doc in docs:
        if hasattr(doc, "page_content"):
            context_texts.append(doc.page_content)
        elif isinstance(doc, dict) and "page_content" in doc:
            context_texts.append(doc["page_content"])
        else:
            context_texts.append(str(doc))
    context = "\n".join(context_texts)

    prompt = f"""
Bạn là một trợ lý thông minh. Trả lời câu hỏi dựa trên ngữ cảnh sau.
Nếu không đủ thông tin, hãy nói rõ điều đó, không được tự bịa đặt.

Ngữ cảnh:
{context}

Câu hỏi: {question}
Trả lời ngắn gọn và chính xác nhất có thể:
"""

    response = llm.invoke(prompt)
    generated_text = response.content
    return generated_text

In [6]:
question = "Theo quy định hiện hành, mức thuế suất thuế giá trị gia tăng (GTGT) áp dụng cho hàng hóa, dịch vụ là bao nhiêu phần trăm?"
generated_answer = rag_query(question)
print("Generated Answer:", generated_answer)

Generated Answer: <think>
Hmm, người dùng đang hỏi về mức thuế suất GTGT theo quy định hiện hành. Nhưng trong ngữ cảnh cung cấp, toàn bộ văn bản tập trung vào việc quản lý thu nhập từ xử lý tài sản tịch thu ở tỉnh Tuyên Quang.

Có vẻ đây là một câu hỏi bị nhầm lẫn giữa các quy định khác nhau. Văn bản này nói về bồi dưỡng làm thêm giờ cho cán bộ công chức và hiệu lực của nó bắt đầu từ năm 2012, trong khi thuế GTGT có những thay đổi đáng kể gần đây hơn.

Mình cần phải cẩn thận vì không được tự bịa đặt thông tin. Trong văn bản này hoàn toàn không đề cập đến thuế GTGT. Mình nên giải thích rõ ràng rằng câu trả lời dựa trên kiến thức chung về luật, chứ không phải từ ngữ cảnh đã cho.

Thuế GTGT hiện hành là 10% và 5%, nhưng đây là kiến thức tổng quát của Việt Nam, không liên quan đến văn bản cụ thể trong ngữ cảnh. Mình sẽ nêu rõ nguồn gốc thông tin này để tránh gây nhầm lẫn.
</think>
Theo quy định hiện hành (tính đến
